## 2 序列模型

### 2.1 理论计算题

给定字符序列 "ababc"，采用一阶马尔可夫模型 p(x_t | x_{t-1})，词汇表为 {a, b, c}，使用拉普拉斯平滑（加1平滑）估计：

1. p(a | b)
2. p(c | b)

**解答：**

统计所有转移（包括未出现）：
- 序列中相邻对：(a,b), (b,a), (a,b), (b,c)（共4个转移）
- 从 b 出发的转移：(b,a) 出现1次，(b,c) 出现1次，(b,b) 出现0次，只考虑目标词为 {a,b,c}。

拉普拉斯平滑公式：
p(x_t = v | x_{t-1} = u) = ( count(u -> v) + 1 ) / ( count(u -> 任何词) + |V| )
其中 |V| = 3。

从 b 出发的总转移数 = 2（(b,a) 和 (b,c)）。

1. p(a | b) = (1 + 1) / (2 + 3) = 2/5 = 0.4

2. p(c | b) = (1 + 1) / (2 + 3) = 2/5 = 0.4

（注意：p(b | b) = (0+1)/5 = 0.2，三项之和为1。）

In [3]:
# 2.2 编程题：文本预处理与滑动窗口
import re
from collections import Counter

def preprocess_text(text, n):
    """
    1. 转小写，去标点（保留字母和空格）
    2. 按空格分词
    3. 构建词汇表（按频率排序，ID从0开始）
    4. 滑动窗口生成特征序列和标签序列（忽略无后续词的窗口）
    返回: vocab_dict, (features, labels)
    """
    # 1. 小写，去除非字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留小写字母和空格
    # 2. 分词（多个空格合并）
    words = text.split()
    
    # 3. 词汇表按频率排序
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))  # 频率降序，同频按字母
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口
    features = []
    labels = []
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
    # 如果 n >= len(words)，则features为空，labels为空，忽略无后续词的窗口（即最后一个窗口无标签则不生成）
    
    return vocab, (features, labels)

# 测试
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, (feat, lab) = preprocess_text(text, n)
    print("词汇表:", vocab)
    print("特征:", feat)
    print("标签:", lab)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3 循环神经网络

### 3.1 理论计算题

考虑线性RNN（无偏置）：
h_t = W_hh * h_{t-1} + W_hx * x_t,   o_t = W_oh * h_t
损失函数 L = 1/2 * sum_{t=1}^{T} (o_t - y_t)^2。

推导损失对 W_hh 的梯度（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

**解答：**

记 e_t = o_t - y_t，则 ∂L/∂o_t = e_t。

根据链式法则，损失对 W_hh 的梯度为所有时间步贡献之和：
∂L/∂W_hh = sum_{t=1}^{T} (∂L/∂h_t) * (∂h_t/∂W_hh)
其中 ∂L/∂h_t 包含当前时刻和未来时刻的梯度传播：
∂L/∂h_t = (∂o_t/∂h_t)^T * (∂L/∂o_t) + (∂h_{t+1}/∂h_t)^T * (∂L/∂h_{t+1})
         = W_oh^T * e_t + W_hh^T * (∂L/∂h_{t+1}), (t<T)
且 ∂L/∂h_T = W_oh^T * e_T。

展开至所有时间步，得到：
∂L/∂W_hh = sum_{t=1}^{T} sum_{k=0}^{T-t} ( prod_{i=0}^{k-1} W_hh^T ) * W_oh^T * e_{t+k} * h_{t-1}^T
（严格说，∂h_t/∂W_hh 的项为 h_{t-1} 加上后续累积）。

**梯度消失或爆炸条件**：
- 若 W_hh 的谱半径（特征值绝对值最大值）小于1，则 prod W_hh^T 随 k 增大指数衰减 → 梯度消失。
- 若谱半径大于1，则指数增长 → 梯度爆炸。
- 当谱半径等于1时，梯度稳定（但实际非线性激活函数会改变条件）。

In [6]:
# 3.2 编程题：RNN单元前向与反向（tanh激活）
import torch
import torch.nn.functional as F

def rnn_step_forward(x, h_prev, W_hh, W_xh, b_h):
    """
    前向传播：计算当前隐藏状态
    x: (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hh: (hidden_size, hidden_size)
    W_xh: (input_size, hidden_size)
    b_h: (hidden_size,)
    返回: h_next (batch_size, hidden_size)
    """
    h_next = torch.tanh(x @ W_xh + h_prev @ W_hh + b_h)
    return h_next

def rnn_step_backward(dh_next, x, h_prev, W_hh, W_xh, b_h, h_next):
    """
    反向传播：已知 dh_next (损失对 h_next 的梯度)
    计算 dx, dh_prev, dW_hh, dW_xh, db_h
    所有梯度形状与对应变量相同
    """
    dtanh = 1 - h_next ** 2
    dh = dh_next * dtanh   # (batch, hidden)
    
    db_h = dh.sum(dim=0)   # (hidden,)
    dW_xh = x.T @ dh       # (input, hidden)
    dW_hh = h_prev.T @ dh  # (hidden, hidden)
    
    dx = dh @ W_xh.T       # (batch, input)
    dh_prev = dh @ W_hh.T  # (batch, hidden)
    
    return dx, dh_prev, dW_xh, dW_hh, db_h

# 测试（修正：x 和 h_prev 也开启梯度）
if __name__ == "__main__":
    batch_size, input_size, hidden_size = 2, 4, 3
    # 让所有输入张量都 requires_grad=True
    x = torch.randn(batch_size, input_size, requires_grad=True)
    h_prev = torch.randn(batch_size, hidden_size, requires_grad=True)
    W_hh = torch.randn(hidden_size, hidden_size, requires_grad=True)
    W_xh = torch.randn(input_size, hidden_size, requires_grad=True)
    b_h = torch.randn(hidden_size, requires_grad=True)
    
    # 前向
    h_next = rnn_step_forward(x, h_prev, W_hh, W_xh, b_h)
    # 模拟上游梯度
    dh_next = torch.randn_like(h_next)
    
    # 手动反向
    dx, dh_prev, dW_xh, dW_hh, db_h = rnn_step_backward(dh_next, x, h_prev, W_hh, W_xh, b_h, h_next)
    
    # 用PyTorch自动梯度验证
    h_next_pt = torch.tanh(x @ W_xh + h_prev @ W_hh + b_h)
    loss = (h_next_pt * dh_next).sum()  # 标量，使 loss 对 h_next_pt 的梯度为 dh_next
    loss.backward()
    
    print("dx 接近?", torch.allclose(dx, x.grad, atol=1e-6))
    print("dh_prev 接近?", torch.allclose(dh_prev, h_prev.grad, atol=1e-6))
    print("dW_xh 接近?", torch.allclose(dW_xh, W_xh.grad, atol=1e-6))
    print("dW_hh 接近?", torch.allclose(dW_hh, W_hh.grad, atol=1e-6))
    print("db_h 接近?", torch.allclose(db_h, b_h.grad, atol=1e-6))

dx 接近? True
dh_prev 接近? True
dW_xh 接近? True
dW_hh 接近? True
db_h 接近? True


## 4 高级循环神经网络

### 4.1 理论计算题

假设一个深度双向RNN，有 L 层，每层隐藏单元数为 H，输入维度为 D，输出维度为 O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

**解答：**

每一层双向RNN包含两个方向的RNN（前向和后向），每个方向的RNN都有：
- 输入到隐藏的权重：D * H（第一层输入为D，其他层输入为2H因为拼接）
- 隐藏到隐藏的权重：H * H
- 偏置：H

但需注意：对于第一层，输入维度为D，对于第 l>1 层，输入维度为 2H（因为前向和后向隐藏状态拼接后作为下一层的输入）。另外，每个方向独立。

计算每一层的参数：

**第1层（l=1）**：
- 前向RNN：输入到隐藏 D*H，隐藏到隐藏 H^2，偏置 H → 总计 D*H + H^2 + H
- 后向RNN：同样 D*H + H^2 + H
- 该层合计：2 * (D*H + H^2 + H)

**第 l 层（2 ≤ l ≤ L）**：
- 输入维度为 2H（前一层两个方向的拼接）
- 前向RNN：输入到隐藏 (2H)*H = 2H^2，隐藏到隐藏 H^2，偏置 H → 总计 3H^2 + H
- 后向RNN：同样 3H^2 + H
- 该层合计：2 * (3H^2 + H) = 6H^2 + 2H

**输出层**（从最后一层隐藏状态到输出 O）：
- 最后一层隐藏状态是前向和后向拼接，维度 2H，输出维度 O，权重 2H * O，偏置 O → 参数 2*H*O + O

**总参数**：
Total = 2*(D*H + H^2 + H) + (L-1)*(6H^2 + 2H) + (2*H*O + O)

简化：
= 2*D*H + 2H^2 + 2H + (L-1)*(6H^2 + 2H) + 2*H*O + O
= 2*D*H + [2 + 6(L-1)]H^2 + [2 + 2(L-1)]H + 2*H*O + O
= 2*D*H + (6L - 4)H^2 + 2L*H + 2*H*O + O

In [7]:
# 4.2 编程题：双向RNN编码器
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,          # 输入形状 (seq_len, batch, input_dim)
            bidirectional=True
        )
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        返回:
            outputs: (seq_len, batch, 2*hidden_dim) 每个时间步拼接的前后向隐藏状态
            final_state: (batch, 2*hidden_dim) 最后一个时间步的拼接状态（双向最后时刻）
        """
        outputs, h_n = self.rnn(X)   # outputs: (seq_len, batch, 2*hidden)
                                     # h_n: (num_layers*2, batch, hidden)
        # 拼接最后一个时间步的前向和后向隐藏状态（双向各自最后时刻）
        # 对于前向，最后时刻是 seq_len-1；后向最后时刻是 0
        # h_n 中顺序：前向层0, 后向层0, 前向层1, 后向层1, ...
        # 取最后一层的前向和后向
        forward_last = h_n[-2, :, :]   # (batch, hidden)
        backward_last = h_n[-1, :, :]  # (batch, hidden)
        final_state = torch.cat([forward_last, backward_last], dim=1)  # (batch, 2*hidden)
        
        return outputs, final_state

# 测试
if __name__ == "__main__":
    seq_len, batch, input_dim = 5, 3, 10
    hidden_dim = 4
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    outputs, final = encoder(X)
    print("outputs shape:", outputs.shape)        # (5, 3, 8)
    print("final state shape:", final.shape)     # (3, 8)

outputs shape: torch.Size([5, 3, 8])
final state shape: torch.Size([3, 8])


## 5 嵌入向量

### 5.1 理论计算题

在Skip-gram模型中，给定中心词 w_c 和上下文词 w_o，使用负采样（采样 K 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 v_c, u_o，负样本词向量为 u_{n_k}，写出完整的目标函数。

**解答：**

Skip-gram 负采样损失函数（对每个训练样本）为：
L = - log σ(u_o^T * v_c) - sum_{k=1}^{K} log σ(-u_{n_k}^T * v_c)
其中 σ(x) = 1/(1+exp(-x)) 是 sigmoid 函数。

第一项鼓励中心词与真实上下文词的点积大（概率高），第二项鼓励与负样本点积小。

**负样本采样**：从噪声分布 P_n(w) 中采样 K 个词（不包含当前上下文词）。常用噪声分布为词频的 3/4 次方（unigram distribution raised to 3/4 power），即
P_n(w) = count(w)^(3/4) / sum_{v} count(v)^(3/4)
这样可以降低高频词的采样概率，提高低频词被采到的机会。

In [8]:
# 5.2 编程题：CBOW（完整softmax）
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_indices, W_in, W_out):
    """
    context_indices: (batch_size, context_size) 每个样本的上下文词索引
    target_indices: (batch_size,) 目标中心词索引
    W_in: (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重矩阵
    返回: 交叉熵损失值 (标量)
    """
    batch_size, context_size = context_indices.shape
    # 获取上下文词向量 (batch, context_size, d)
    embeds = W_in[context_indices]   # (batch, context_size, d)
    # 平均上下文向量作为隐藏层 (batch, d)
    h = embeds.mean(dim=1)           # (batch, d)
    # 计算得分 (batch, V)
    scores = h @ W_out               # (batch, V)
    # 计算交叉熵损失（目标为中心词索引）
    loss = F.cross_entropy(scores, target_indices)
    return loss

# 测试
if __name__ == "__main__":
    V, d = 10, 5
    batch_size, context_size = 4, 3
    W_in = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    context = torch.randint(0, V, (batch_size, context_size))
    targets = torch.randint(0, V, (batch_size,))
    loss = cbow_forward(context, targets, W_in, W_out)
    print("CBOW损失值:", loss.item())

CBOW损失值: 2.1842122077941895


## 6 注意力机制

### 6.1 理论计算题

给定查询矩阵 Q ∈ R^(2×4)，键矩阵 K ∈ R^(3×4)，值矩阵 V ∈ R^(3×5)。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（得分矩阵、softmax、加权求和）。使用 score = Q * K^T / sqrt(d_k)，d_k = 4。可列出数值计算过程（符号或具体数值）。

**解答：**

假设具体数值（为展示过程，使用符号表示）：

设
Q = [q11, q12, q13, q14; q21, q22, q23, q24],
K = [k11, k12, k13, k14; k21, k22, k23, k24; k31, k32, k33, k34],
V = [v11, ..., v15; v21, ..., v25; v31, ..., v35]

**得分矩阵**（2×3）：
S = Q * K^T / sqrt(4) = 1/2 * Q * K^T
其中 S_ij = 1/2 * sum_{r=1}^{4} Q_{i r} * K_{j r}。

**Softmax按行**（对每个查询）：
A_ij = exp(S_ij) / sum_{l=1}^{3} exp(S_il)
得到注意力权重矩阵 A ∈ R^(2×3)。

**输出矩阵**（2×5）：
Output = A * V = [
  sum_{j=1}^{3} A_1j * V_{j1} , ..., sum_{j=1}^{3} A_1j * V_{j5} ;
  sum_{j=1}^{3} A_2j * V_{j1} , ..., sum_{j=1}^{3} A_2j * V_{j5}
]
即每个查询对应的输出是值向量的加权和。

In [9]:
# 6.2 编程题：多头注意力（PyTorch实现）
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层，用于Q、K、V，以及最终输出
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        返回: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 分头: 调整为 (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_k)
        
        # 交换维度，使得 (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)  # (batch, heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (batch, heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (batch, heads, seq_len, d_k)
        
        # 合并头: (batch, seq_len, heads*d_k) -> (seq_len, batch, d_model)
        attn_output = attn_output.permute(2, 0, 1, 3).contiguous()  # (seq_len, batch, heads, d_k)
        attn_output = attn_output.view(seq_len, batch, -1)          # (seq_len, batch, d_model)
        
        # 最终线性层
        output = self.W_o(attn_output)
        return output

# 测试
if __name__ == "__main__":
    d_model = 4
    num_heads = 2
    seq_len, batch = 6, 3
    X = torch.randn(seq_len, batch, d_model)
    mha = MultiHeadAttention(d_model, num_heads)
    out = mha(X)
    print("输入形状:", X.shape)
    print("输出形状:", out.shape)
    assert out.shape == X.shape, "输出形状应与输入相同"
    print("测试通过！")

输入形状: torch.Size([6, 3, 4])
输出形状: torch.Size([6, 3, 4])
测试通过！
